# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to explore, process, and analyze the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset Croissant schema is available at:  
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure mlcroissant is installed.
!pip install mlcroissant

## 1. Data Loading

Load and inspect the dataset metadata from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show the main title and description
meta = dataset.metadata
print(f"{meta.name}\n\n{meta.description}\n")

## 2. Data Overview

List all available record sets, with their `@id` and names, as well as associated fields and columns. This will help determine what data is available for extraction.

In [ ]:
# List all record sets by @id and name
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}    name: {rs.get('name', '<no name>')}")

# For each record set, list the fields and columns by their @id
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs and rs['field']:
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"    @id: {f['@id']}    name: {f.get('name', '<no name>')}")
            else:
                print(f"    @id: {f}")
    if 'column' in rs and rs['column']:
        print("  Columns:")
        for c in rs['column']:
            if isinstance(c, dict):
                print(f"    @id: {c['@id']}    name: {c.get('name', '<no name>')}")
            else:
                print(f"    @id: {c}")

## 3. Data Extraction

Load all records from each available record set into a pandas DataFrame for analysis. Use the record set `@id`s from above.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
# Prepare DataFrames
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if len(records) > 0:
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set '{rsid}'. Columns:")
        print(dataframes[rsid].columns.tolist())
        print(dataframes[rsid].head(), "\n")
    else:
        print(f"No records present for record set '{rsid}'.")

## 4. Exploratory Data Analysis (EDA)

As an example, let's select a numeric field from one record set (by its `@id`), filter and normalize the data, then group by a categorical field if present.

_**Note:** Use the `@id` of fields as listed in section 2 above. If the dataset does not contain any record sets with data, update this cell after record set and field `@id`s are known._

In [ ]:
# For this exploration, we assume there's at least one DataFrame with a numeric field.
# Replace these variables with actual @id values as discovered above, e.g.:
#   selected_record_set_id = 'cr:regression_results'
#   numeric_field_id = 'cr:log_likelihood'
#   group_field_id = 'cr:county'

# If you know the exact record set and field IDs from the previous step, update these variables accordingly:
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    # Attempt to automatically find a numeric field
    numeric_field_id = None
    example_numeric = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            example_numeric = df[col]
            break
    if numeric_field_id:
        threshold = example_numeric.mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_field]].head())

        # Attempt to find a categorical/text field for grouping
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No categorical/text field found for grouping.")
    else:
        print("No numeric field detected in the selected record set.")
else:
    print("No populated record sets available for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field, and relationships to a categorical variable if present.

_**Tip:** You may need to install matplotlib or seaborn. The following code will only execute if EDA has produced data._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if possible
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to load, inspect, and analyze a dataset defined by a Croissant schema.

- Record sets and fields were referenced with their `@id` for unambiguous data selection.
- Example data extraction and exploratory steps (filtering, normalization, grouping) were provided using pandas.
- Plots demonstrate statistical distribution and categorical comparison when possible.

You can further extend this workflow for feature engineering, advanced analyses, or exporting curated subsets for downstream modeling.